# AOOP ARMONYX - Notebook 1: Qwen + MusicGen Worker

## Endpoints

```
GET  /health
POST /classify-intent        <- síncrono
POST /generate-plan          <- async 202
POST /generate-audio         <- async 202
POST /generate-all           <- async 202
GET  /status/{job_id}
GET  /result/{job_id}/plan
GET  /result/{job_id}/audio  <- StreamingResponse WAV binário
```

## Notas
- Qwen2.5:14b corre no Ollama, GPU 0.
- MusicGen corre via transformers, GPU 1.
- Secrets via Kaggle Secrets.
- Sem escrita em disco — áudio gerado em memória.

## 1 - Instalar dependências

In [ ]:
import os, sys, subprocess

def run(cmd):
    print(">>>", " ".join(cmd))
    result = subprocess.run(cmd)
    if result.returncode != 0:
        raise RuntimeError(f"Comando falhou: {' '.join(cmd)}")

subprocess.run(["apt-get", "update", "-qq"], check=False)
subprocess.run(["apt-get", "install", "-y", "-qq", "zstd", "lshw", "pciutils", "ffmpeg", "curl"], check=True)
subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, check=True)
run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "pip", "setuptools", "wheel"])
run([sys.executable, "-m", "pip", "install", "-q",
     "fastapi", "uvicorn[standard]", "pyngrok",
     "requests", "httpx", "pydantic",
     "scipy", "numpy", "soundfile",
     "accelerate", "safetensors", "sentencepiece", "huggingface_hub"])
run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "transformers"])
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchvision"], check=False)
print("Dependências instaladas correctamente.")

## 2 - Imports e configuração

In [ ]:
import io, os, re, json, time, uuid, threading, subprocess
from datetime import datetime, timedelta
from pathlib import Path
from typing import Optional, Dict, Any

import numpy as np
import requests
import torch
import scipy.io.wavfile

from fastapi import FastAPI, HTTPException, Header, BackgroundTasks
from fastapi.responses import StreamingResponse
from pydantic import BaseModel, Field
from pyngrok import ngrok, conf
from transformers import AutoProcessor, MusicgenForConditionalGeneration
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

# ── Secrets
_secrets       = UserSecretsClient()
NGROK_TOKEN    = _secrets.get_secret("NGROK_TOKEN")
NGROK_DOMAIN   = _secrets.get_secret("NGROK_DOMAIN")
HF_TOKEN       = _secrets.get_secret("HF_TOKEN")
WORKER_API_KEY = _secrets.get_secret("WORKER_API_KEY")
login(token=HF_TOKEN)
print("Hugging Face autenticado.")

# ── Pastas
MODELS_DIR = Path("/kaggle/working/ollama_models")
MODELS_DIR.mkdir(parents=True, exist_ok=True)
os.environ["OLLAMA_MODELS"] = str(MODELS_DIR)

# ── Configuração
QWEN_MODEL            = "qwen2.5:14b"
MUSICGEN_MODEL_NAME   = "facebook/musicgen-medium"
DEFAULT_NUM_SCENES    = 12
DEFAULT_MUSIC_DURATION = 60
MAX_MUSIC_DURATION    = 120
PORT                  = 8000
JOB_TTL_MINUTES       = 30

# ── Dispositivos
QWEN_GPU        = "0"
MUSICGEN_DEVICE = "cuda:1" if torch.cuda.device_count() > 1 else "cuda:0"

print(f"PyTorch: {torch.__version__}")
print(f"CUDA disponível: {torch.cuda.is_available()}")
print(f"GPUs disponíveis: {torch.cuda.device_count()}")
print(f"MusicGen device: {MUSICGEN_DEVICE}")
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(f"GPU {i}: {props.name} — {props.total_memory // 1024**2} MB")

## 3 - Iniciar Ollama na GPU 0

In [ ]:
def stop_ollama():
    subprocess.run(["killall", "ollama"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    time.sleep(2)

def start_ollama():
    env = os.environ.copy()
    env["PATH"]                = env.get("PATH", "") + ":/usr/local/bin"
    env["OLLAMA_MODELS"]       = str(MODELS_DIR)
    env["OLLAMA_HOST"]         = "0.0.0.0:11434"
    env["OLLAMA_ORIGINS"]      = "*"
    env["CUDA_VISIBLE_DEVICES"] = "0"
    process = subprocess.Popen(
        ["/usr/local/bin/ollama", "serve"],
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, env=env,
    )
    for attempt in range(24):
        time.sleep(5)
        try:
            r = requests.get("http://localhost:11434/api/tags", timeout=3)
            if r.status_code == 200:
                print("Ollama iniciado.")
                return process
        except Exception:
            pass
        print(f"A aguardar Ollama... {(attempt + 1) * 5}s")
    raise RuntimeError("Ollama não iniciou dentro do tempo esperado.")

stop_ollama()
ollama_process = start_ollama()
get_ipython().system("nvidia-smi")

## 4 - Descarregar e testar Qwen

In [ ]:
def run_shell(command):
    return subprocess.run(command, shell=True).returncode

print(f"A descarregar {QWEN_MODEL}...")
if run_shell(f"OLLAMA_MODELS={MODELS_DIR} ollama pull {QWEN_MODEL}") != 0:
    raise RuntimeError(f"Erro ao descarregar {QWEN_MODEL}")
print(f"{QWEN_MODEL} pronto.")

r = requests.post(
    "http://localhost:11434/api/chat",
    json={"model": QWEN_MODEL, "messages": [{"role": "user", "content": "Responde apenas: OK"}], "stream": False},
    timeout=180,
)
r.raise_for_status()
print("Teste Qwen:", r.json().get("message", {}).get("content", "SEM_RESPOSTA"))

## 5 - Carregar MusicGen na GPU 1

In [ ]:
if torch.cuda.device_count() < 2:
    raise RuntimeError("Este notebook espera 2 GPUs. No Kaggle, selecciona Accelerator: GPU T4 x2.")

MUSICGEN_DEVICE = "cuda:1"
print(f"A carregar {MUSICGEN_MODEL_NAME} em {MUSICGEN_DEVICE}...")

music_processor = AutoProcessor.from_pretrained(MUSICGEN_MODEL_NAME)
musicgen_model  = MusicgenForConditionalGeneration.from_pretrained(
    MUSICGEN_MODEL_NAME, torch_dtype=torch.float16
).to(MUSICGEN_DEVICE)
musicgen_model.eval()

print(f"MusicGen carregado em {MUSICGEN_DEVICE}.")
for i in range(torch.cuda.device_count()):
    used  = torch.cuda.memory_allocated(i) // 1024**2
    total = torch.cuda.get_device_properties(i).total_memory // 1024**2
    print(f"GPU {i}: {used} MB / {total} MB")

## 6 - Funções auxiliares e store de jobs

In [ ]:
# ── Store de jobs em memória
jobs: Dict[str, Dict[str, Any]] = {}
jobs_lock = threading.Lock()

def create_job(job_id: str) -> str:
    with jobs_lock:
        jobs[job_id] = {"status": "queued", "progress": 0, "result": None, "error": None, "created_at": datetime.now()}
    return job_id

def update_job(job_id: str, **kwargs) -> None:
    with jobs_lock:
        if job_id in jobs:
            jobs[job_id].update(kwargs)

def cleanup_expired_jobs() -> None:
    cutoff = datetime.now() - timedelta(minutes=JOB_TTL_MINUTES)
    with jobs_lock:
        expired = [jid for jid, j in jobs.items() if j["created_at"] < cutoff]
        for jid in expired:
            jobs.pop(jid, None)
    if expired:
        print(f"[cleanup] {len(expired)} jobs expirados removidos.")

# ── Validação
def require_worker_key(x_worker_key: Optional[str]) -> None:
    if WORKER_API_KEY and x_worker_key != WORKER_API_KEY:
        raise HTTPException(status_code=401, detail="Chave do worker inválida.")

def safe_job_id(job_id: str) -> str:
    cleaned = re.sub(r"[^a-zA-Z0-9_\-]", "_", job_id.strip())
    if not cleaned:
        raise HTTPException(status_code=400, detail="job_id inválido.")
    return cleaned

# ── JSON helpers
def extract_json_object(text: str) -> Dict[str, Any]:
    text = text.strip()
    if text.startswith("```"):
        text = re.sub(r"^```(?:json)?", "", text, flags=re.IGNORECASE).strip()
        text = re.sub(r"```$", "", text).strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass
    start, end = text.find("{"), text.rfind("}")
    if start == -1 or end == -1 or end <= start:
        raise ValueError("A resposta não contém um objecto JSON.")
    return json.loads(text[start:end + 1])

# ── Áudio em memória
def normalise_audio_to_int16(audio_data: np.ndarray) -> np.ndarray:
    audio_data = np.nan_to_num(audio_data)
    max_abs = float(np.max(np.abs(audio_data))) if audio_data.size else 0.0
    if max_abs > 0:
        audio_data = audio_data / max_abs
    return (audio_data * 32767.0).astype(np.int16)

def generate_audio_bytes(music_prompt: str, duration_seconds: int = DEFAULT_MUSIC_DURATION):
    if music_processor is None or musicgen_model is None:
        raise RuntimeError("MusicGen ainda não foi carregado.")
    duration_seconds = int(max(5, min(duration_seconds, MAX_MUSIC_DURATION)))
    frame_rate       = musicgen_model.config.audio_encoder.frame_rate
    max_new_tokens   = int(duration_seconds * frame_rate)

    inputs = music_processor(text=[music_prompt], padding=True, return_tensors="pt")
    inputs = {k: v.to(MUSICGEN_DEVICE) for k, v in inputs.items()}

    with torch.no_grad():
        audio_values = musicgen_model.generate(
            **inputs, max_new_tokens=max_new_tokens, do_sample=True, guidance_scale=3.0
        )

    audio       = audio_values[0, 0].detach().cpu().float().numpy()
    audio_int16 = normalise_audio_to_int16(audio)
    sample_rate = musicgen_model.config.audio_encoder.sampling_rate

    buf = io.BytesIO()
    scipy.io.wavfile.write(buf, sample_rate, audio_int16)
    wav_bytes = buf.getvalue()

    metadata = {
        "music_prompt": music_prompt, "duration_seconds": duration_seconds,
        "sample_rate": sample_rate, "model": MUSICGEN_MODEL_NAME,
        "device": MUSICGEN_DEVICE, "max_new_tokens": max_new_tokens,
        "size_bytes": len(wav_bytes), "created_at": datetime.now().isoformat(),
    }
    return wav_bytes, metadata

# ── Prompts
def build_plan_prompt(theme: str, style: str, num_scenes: int, music_duration: int) -> str:
    return (
        f'Cria um plano criativo completo para um AMV com o tema: "{theme}"\n'
        f'Estilo pedido pelo utilizador: "{style}"\n\n'
        "Responde APENAS com JSON válido, sem markdown e sem texto fora do JSON.\n"
        "Usa Português de Portugal nas descrições e na letra.\n"
        "Os `image_prompt` e `music_prompt` devem estar em inglês.\n\n"
        "Formato obrigatório:\n"
        "{\n"
        '  "title": "título do AMV",\n'
        f'  "theme": "{theme}",\n'
        f'  "style": "{style}",\n'
        '  "lyrics": [{"line": "verso 1", "timestamp": 0.0}],\n'
        '  "music_prompt": "detailed English prompt for instrumental anime music",\n'
        '  "storyboard": [{\n'
        '    "scene_index": 1,\n'
        '    "description": "descrição em Português de Portugal",\n'
        '    "image_prompt": "detailed anime image prompt in English for SDXL",\n'
        '    "duration": 5.0,\n'
        '    "transition": "fade"\n'
        "  }]\n"
        "}\n\n"
        f"Regras:\n"
        f"- Gera exactamente {num_scenes} cenas.\n"
        f"- Duração total: aproximadamente {music_duration} segundos.\n"
        "- image_prompt e music_prompt em inglês.\n"
        "- Música instrumental, energética, adequada a AMV anime.\n"
        "- Não uses personagens protegidas por copyright."
    )

def normalise_creative_plan(plan: Dict[str, Any], theme: str, style: str, num_scenes: int) -> Dict[str, Any]:
    plan.setdefault("title", "AMV Gerado")
    plan.setdefault("theme", theme)
    plan.setdefault("style", style)
    plan.setdefault("music_prompt", "epic anime instrumental, emotional cinematic rock, energetic drums")
    if not isinstance(plan.get("lyrics"), list):
        plan["lyrics"] = []
    if not isinstance(plan.get("storyboard"), list):
        plan["storyboard"] = []
    storyboard = plan["storyboard"][:num_scenes]
    while len(storyboard) < num_scenes:
        idx = len(storyboard) + 1
        storyboard.append({
            "scene_index": idx,
            "description": f"Cena {idx} inspirada no tema principal.",
            "image_prompt": f"anime cinematic scene inspired by {theme}, dramatic lighting, detailed",
            "duration": 5.0, "transition": "fade",
        })
    for idx, scene in enumerate(storyboard, start=1):
        scene["scene_index"] = idx
        scene.setdefault("description", f"Cena {idx}")
        scene.setdefault("image_prompt", f"anime cinematic scene, {theme}, dramatic lighting")
        scene.setdefault("duration", 5.0)
        scene.setdefault("transition", "fade")
    plan["storyboard"] = storyboard
    return plan

def build_intent_prompt(message: str, context: dict) -> str:
    # Estado do projecto
    state = {
        "title":            context.get("title"),
        "current_status":   context.get("current_status", "idle"),
        "generation_phase": context.get("generation_phase", "idle"),
        "has_plan":         context.get("has_plan", False),
        "has_audio":        context.get("has_audio", False),
        "has_images":       context.get("has_images", False),
        "has_video":        context.get("has_video", False),
    }

    # Música gerada anteriormente — contexto chave para refinação de prompts MusicGen
    music_context = ""
    if context.get("music_prompt"):
        settings = context.get("settings") or {}
        music_context = (
            "\nMúsica gerada anteriormente:\n"
            f'  music_prompt: "{context["music_prompt"]}"\n'
            f'  genre: {settings.get("genre", "—")}\n'
            f'  tempo: {settings.get("tempo", "—")}\n'
            f'  mood: {settings.get("mood", "—")}\n'
            f'  duration: {settings.get("duration", "—")}\n'
        )

    # Histórico da conversa (últimas 8 msgs) para contexto
    history_text = ""
    history = context.get("conversation_history", [])
    if history:
        lines = [f'  {m["role"].upper()}: {m["content"]}' for m in history[-8:]]
        history_text = "\nHistórico da conversa:\n" + "\n".join(lines) + "\n"

    return (
        "És o assistente criativo do ARMONYX, um gerador de AMVs anime.\n"
        "Respondes sempre em Português de Portugal.\n\n"
        f"Estado actual do projecto:\n{json.dumps(state, ensure_ascii=False)}\n"
        f"{music_context}"
        f"{history_text}\n"
        f'Mensagem do utilizador: "{message}"\n\n'
        "Classifica a intenção e responde APENAS com JSON válido, sem markdown.\n\n"
        "Formato obrigatório:\n"
        "{\n"
        '  "intent": "chat|plan|audio|images|video|regenerate_audio|regenerate_images|regenerate_video",\n'
        '  "params": {},\n'
        '  "response_text": "resposta em Português de Portugal para mostrar ao utilizador"\n'
        "}\n\n"
        "Regras de classificação:\n"
        '- "chat": dúvida, sugestão criativa, pergunta geral\n'
        '- "plan": quer criar/definir o plano criativo\n'
        '- "audio": quer gerar só a música\n'
        '- "images": quer gerar só as imagens\n'
        '- "video": quer o videoclip completo\n'
        '- "regenerate_audio": quer refazer/ajustar a música existente\n'
        '- "regenerate_images": quer refazer imagens\n'
        '- "regenerate_video": quer remontar o vídeo\n\n'
        "Regras para params em regenerate_audio:\n"
        "- OBRIGATÓRIO incluir 'music_prompt' com o prompt anterior MELHORADO segundo o pedido do utilizador\n"
        "- O music_prompt deve estar em inglês, detalhado e específico para MusicGen\n"
        "- Incorporar os ajustes pedidos (ex: 'mais energético' → adicionar 'energetic, fast tempo 160bpm, intense drums, powerful brass')\n"
        "- Manter os elementos que o utilizador não pediu para mudar\n"
        "- Incluir: genre, tempo, mood, duration se mencionados ou relevantes\n\n"
        "Em params inclui sempre: theme, style, duration, music_prompt (se áudio), scene_index (se imagem), etc."
    )

print("Funções auxiliares e store de jobs definidos.")

## 7 - FastAPI app

In [ ]:
app = FastAPI(title="AOOP ARMONYX Worker 1 — Qwen + MusicGen")

# ── Modelos de pedido
class ClassifyIntentRequest(BaseModel):
    message: str  = Field(..., min_length=1)
    context: dict = Field(default_factory=dict)

class GeneratePlanRequest(BaseModel):
    job_id:         str = Field(..., min_length=1)
    theme:          str = Field(..., min_length=3)
    style:          str = Field("anime cinematic emotional")
    num_scenes:     int = Field(DEFAULT_NUM_SCENES, ge=1, le=24)
    music_duration: int = Field(DEFAULT_MUSIC_DURATION, ge=5, le=MAX_MUSIC_DURATION)

class GenerateAudioRequest(BaseModel):
    job_id:           str           = Field(..., min_length=1)
    music_prompt:     str           = Field(..., min_length=3)
    duration_seconds: Optional[int] = None
    def resolved_duration(self):
        v = self.duration_seconds if self.duration_seconds is not None else DEFAULT_MUSIC_DURATION
        return int(max(5, min(v, MAX_MUSIC_DURATION)))

class GenerateAllRequest(BaseModel):
    job_id:           str           = Field(..., min_length=1)
    theme:            str           = Field(..., min_length=3)
    style:            str           = Field("anime cinematic emotional")
    num_scenes:       int           = Field(DEFAULT_NUM_SCENES, ge=1, le=24)
    duration_seconds: Optional[int] = None
    def resolved_duration(self):
        v = self.duration_seconds if self.duration_seconds is not None else DEFAULT_MUSIC_DURATION
        return int(max(5, min(v, MAX_MUSIC_DURATION)))

# ── Background tasks
def _task_generate_plan(job_id: str, req: GeneratePlanRequest) -> None:
    cleanup_expired_jobs()
    update_job(job_id, status="running", progress=5)
    try:
        system_prompt = (
            "És um criador de AMVs criativo e rigoroso. "
            "Deves responder apenas com JSON válido. "
            "Não uses markdown. Não expliques a resposta. "
            "Usa Português de Portugal."
        )
        user_prompt = build_plan_prompt(req.theme, req.style, req.num_scenes, req.music_duration)
        update_job(job_id, progress=10)

        response = requests.post(
            "http://localhost:11434/api/chat",
            json={
                "model": QWEN_MODEL,
                "messages": [
                    {"role": "system", "content": system_prompt},
                    {"role": "user",   "content": user_prompt},
                ],
                "stream": False, "format": "json",
                "options": {"temperature": 0.8, "num_ctx": 8192},
            },
            timeout=420,
        )
        response.raise_for_status()
        update_job(job_id, progress=80)

        content       = response.json().get("message", {}).get("content", "")
        creative_plan = normalise_creative_plan(
            extract_json_object(content), req.theme, req.style, req.num_scenes)

        update_job(job_id, status="done", progress=100, result={"creative_plan": creative_plan})

    except Exception as e:
        update_job(job_id, status="error", error=str(e))

def _task_generate_audio(job_id: str, req: GenerateAudioRequest) -> None:
    cleanup_expired_jobs()
    update_job(job_id, status="running", progress=5)
    try:
        wav_bytes, metadata = generate_audio_bytes(req.music_prompt, req.resolved_duration())
        metadata["job_id"]  = safe_job_id(req.job_id)
        update_job(job_id, status="done", progress=100,
                   result={"audio_bytes": wav_bytes, "metadata": metadata})
    except Exception as e:
        update_job(job_id, status="error", error=str(e))

def _task_generate_all(job_id: str, req: GenerateAllRequest) -> None:
    cleanup_expired_jobs()
    update_job(job_id, status="running", progress=5)
    try:
        duration = req.resolved_duration()
        system_prompt = (
            "És um criador de AMVs criativo e rigoroso. "
            "Deves responder apenas com JSON válido. "
            "Não uses markdown. Não expliques a resposta. "
            "Usa Português de Portugal."
        )
        response = requests.post(
            "http://localhost:11434/api/chat",
            json={
                "model": QWEN_MODEL,
                "messages": [
                    {"role": "system", "content": system_prompt},
                    {"role": "user",   "content": build_plan_prompt(req.theme, req.style, req.num_scenes, duration)},
                ],
                "stream": False, "format": "json",
                "options": {"temperature": 0.8, "num_ctx": 8192},
            },
            timeout=420,
        )
        response.raise_for_status()
        update_job(job_id, progress=50)

        content       = response.json().get("message", {}).get("content", "")
        creative_plan = normalise_creative_plan(
            extract_json_object(content), req.theme, req.style, req.num_scenes)

        music_prompt = creative_plan.get("music_prompt")
        if not music_prompt:
            raise ValueError("O plano não contém music_prompt.")

        wav_bytes, metadata = generate_audio_bytes(music_prompt, duration)
        metadata["job_id"]  = safe_job_id(req.job_id)

        update_job(job_id, status="done", progress=100, result={
            "creative_plan": creative_plan,
            "audio_bytes":   wav_bytes,
            "metadata":      metadata,
        })
    except Exception as e:
        update_job(job_id, status="error", error=str(e))

# ── Endpoints
@app.get("/health")
def health():
    ollama_ok, qwen_available = False, False
    try:
        r = requests.get("http://localhost:11434/api/tags", timeout=3)
        ollama_ok = r.status_code == 200
        if ollama_ok:
            models = r.json().get("models", [])
            qwen_available = any(QWEN_MODEL in m.get("name", "") for m in models)
    except Exception:
        pass
    gpu_info = [{"index": i, "name": torch.cuda.get_device_name(i),
                 "memory_allocated_mb": torch.cuda.memory_allocated(i) // 1024**2,
                 "memory_total_mb": torch.cuda.get_device_properties(i).total_memory // 1024**2}
                for i in range(torch.cuda.device_count())]
    return {
        "status": "ok", "timestamp": datetime.now().isoformat(),
        "ollama": ollama_ok, "qwen_model": QWEN_MODEL, "qwen_available": qwen_available,
        "musicgen_model": MUSICGEN_MODEL_NAME, "musicgen_loaded": musicgen_model is not None,
        "jobs_in_memory": len(jobs), "gpus": gpu_info,
    }

@app.post("/classify-intent")
def classify_intent(req: ClassifyIntentRequest, x_worker_key: Optional[str] = Header(default=None)):
    require_worker_key(x_worker_key)
    try:
        r = requests.post(
            "http://localhost:11434/api/chat",
            json={
                "model": QWEN_MODEL,
                "messages": [{"role": "user", "content": build_intent_prompt(req.message, req.context)}],
                "stream": False, "format": "json",
                # num_ctx aumentado para 8192 — necessário para histórico + music_prompt no contexto
                "options": {"temperature": 0.3, "num_ctx": 8192},
            },
            timeout=60,
        )
        r.raise_for_status()
        result = extract_json_object(r.json().get("message", {}).get("content", ""))
        result.setdefault("intent", "chat")
        result.setdefault("params", {})
        result.setdefault("response_text", "Como posso ajudar?")
        return result
    except Exception as e:
        raise HTTPException(status_code=500, detail=f"Erro ao classificar intenção: {e}")

@app.post("/generate-plan", status_code=202)
def generate_plan(req: GeneratePlanRequest, background_tasks: BackgroundTasks,
                  x_worker_key: Optional[str] = Header(default=None)):
    require_worker_key(x_worker_key)
    job_id = safe_job_id(req.job_id)
    create_job(job_id)
    background_tasks.add_task(_task_generate_plan, job_id, req)
    return {"job_id": job_id, "status": "queued"}

@app.post("/generate-audio", status_code=202)
def generate_audio(req: GenerateAudioRequest, background_tasks: BackgroundTasks,
                   x_worker_key: Optional[str] = Header(default=None)):
    require_worker_key(x_worker_key)
    job_id = safe_job_id(req.job_id)
    create_job(job_id)
    background_tasks.add_task(_task_generate_audio, job_id, req)
    return {"job_id": job_id, "status": "queued"}

@app.post("/generate-all", status_code=202)
def generate_all(req: GenerateAllRequest, background_tasks: BackgroundTasks,
                 x_worker_key: Optional[str] = Header(default=None)):
    require_worker_key(x_worker_key)
    job_id = safe_job_id(req.job_id)
    create_job(job_id)
    background_tasks.add_task(_task_generate_all, job_id, req)
    return {"job_id": job_id, "status": "queued"}

@app.get("/status/{job_id}")
def get_status(job_id: str, x_worker_key: Optional[str] = Header(default=None)):
    require_worker_key(x_worker_key)
    job = jobs.get(safe_job_id(job_id))
    if not job:
        raise HTTPException(status_code=404, detail="Job não encontrado.")
    return {"job_id": job_id, "status": job["status"], "progress": job["progress"],
            "error": job.get("error"), "has_result": job["result"] is not None}

@app.get("/result/{job_id}/plan")
def result_plan(job_id: str, x_worker_key: Optional[str] = Header(default=None)):
    require_worker_key(x_worker_key)
    job = jobs.get(safe_job_id(job_id))
    if not job:
        raise HTTPException(status_code=404, detail="Job não encontrado.")
    if job["status"] != "done":
        raise HTTPException(status_code=409, detail=f"Job ainda não concluído: {job['status']}")
    result = job.get("result", {})
    if "creative_plan" not in result:
        raise HTTPException(status_code=404, detail="Plano não disponível neste job.")
    return {"job_id": job_id, "creative_plan": result["creative_plan"]}

@app.get("/result/{job_id}/audio")
def result_audio(job_id: str, x_worker_key: Optional[str] = Header(default=None)):
    require_worker_key(x_worker_key)
    job = jobs.get(safe_job_id(job_id))
    if not job:
        raise HTTPException(status_code=404, detail="Job não encontrado.")
    if job["status"] != "done":
        raise HTTPException(status_code=409, detail=f"Job ainda não concluído: {job['status']}")
    result = job.get("result", {})
    if "audio_bytes" not in result:
        raise HTTPException(status_code=404, detail="Áudio não disponível neste job.")
    return StreamingResponse(
        io.BytesIO(result["audio_bytes"]),
        media_type="audio/wav",
        headers={"Content-Disposition": f"attachment; filename=audio_{job_id}.wav"},
    )

print("FastAPI app definida.")

## 8 - Iniciar FastAPI + ngrok

In [ ]:
def start_uvicorn():
    import uvicorn
    uvicorn.run(app, host="0.0.0.0", port=PORT, log_level="info")

ngrok.kill()
time.sleep(1)

server_thread = threading.Thread(target=start_uvicorn, daemon=True)
server_thread.start()
time.sleep(5)

try:
    r = requests.get(f"http://localhost:{PORT}/health", timeout=10)
    r.raise_for_status()
    print("Servidor local iniciado.")
    print(json.dumps(r.json(), indent=2, ensure_ascii=False))
except Exception as e:
    raise RuntimeError(f"Erro ao iniciar FastAPI local: {e}")

conf.get_default().auth_token = NGROK_TOKEN
tunnel       = ngrok.connect(PORT, domain=NGROK_DOMAIN) if NGROK_DOMAIN else ngrok.connect(PORT)
WORKER_1_URL = tunnel.public_url

print("=" * 80)
print("WORKER_1_URL=" + WORKER_1_URL)
print("=" * 80)
for ep in ["/health", "/classify-intent", "/generate-plan",
           "/generate-audio", "/generate-all",
           "/status/{job_id}", "/result/{job_id}/plan", "/result/{job_id}/audio"]:
    print(f"  {WORKER_1_URL}{ep}")

## 9 - Testes rápidos

In [ ]:
import json, requests, time

headers = {"X-Worker-Key": WORKER_API_KEY} if WORKER_API_KEY else {}

# ── Health
r = requests.get(f"{WORKER_1_URL}/health", timeout=20)
print("Health:", r.status_code)
print(json.dumps(r.json(), indent=2, ensure_ascii=False))

# ── Classify intent
r = requests.post(
    f"{WORKER_1_URL}/classify-intent",
    json={"message": "quero uma música de batalha épica", "context": {}},
    headers=headers, timeout=60,
)
print("\nClassify intent:", r.status_code)
print(json.dumps(r.json(), indent=2, ensure_ascii=False))

## 10 - Keep-alive

In [ ]:
print("Worker 1 activo. Mantém esta célula a correr enquanto usares o backend.")
print("-" * 80)

while True:
    try:
        r    = requests.get(f"http://localhost:{PORT}/health", timeout=10)
        data = r.json()
        gpu_text = [
            f"GPU{g['index']}: {g['memory_allocated_mb']}/{g['memory_total_mb']}MB"
            for g in data.get("gpus", [])
        ]
        print(
            f"[{datetime.now().strftime('%H:%M:%S')}] "
            f"ollama={data.get('ollama')} "
            f"qwen={data.get('qwen_available')} "
            f"musicgen={data.get('musicgen_loaded')} "
            f"jobs={data.get('jobs_in_memory', 0)} "
            f"| {' | '.join(gpu_text)}"
        )
    except Exception as e:
        print(f"[{datetime.now().strftime('%H:%M:%S')}] ERRO: {e}")
    time.sleep(30)